In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import unicodedata
import regex as re
import time 
import unicodedata
from collections import Counter
from langdetect import detect
import random
import swifter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
import string
import nltk
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier  
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance

lemmatizer = WordNetLemmatizer()


### Preprocessing


In [2]:
df_original = pd.read_csv(os.path.join("../data/", "train_submission.csv"), encoding='utf-8')

print("Before Cleaning: ",df_original.shape)

df_copy = df_original.copy()
df_copy.rename(columns={'Text': 'text', 'Label': 'target'}, inplace=True)

df_filtered = df_copy[~(df_copy.target.isna())|(df_copy.text.isna())|((df_copy.text == ""))]
print("After Cleaning", df_filtered.shape)

print("Number of unique languages: ",len(df_filtered.target.unique()))

Before Cleaning:  (190599, 3)
After Cleaning (190099, 3)
Number of unique languages:  389


In [3]:
class TextPreprocessor:
    def __init__(self):
        # self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
        self.punct_table = str.maketrans('', '', string.punctuation)
        
    def clean_text(self, text: str) -> str:
        # Combined regex operations
        text = re.sub(r'http\S+|www\S+|@\w+|#\w+|\d+', '', text.lower())
        text = text.translate(self.punct_table)
        return ' '.join([
            self.lemmatizer.lemmatize(word)
            for word in text.split()
            # if word not in self.stop_words
        ])
    
preprocessor = TextPreprocessor()

In [4]:
# remove targets that appear less then 50 times
mask_more_50 = (df_filtered.target.value_counts()).reset_index()
df_filter = df_filtered.merge(mask_more_50, on="target")
df_filter = df_filter[df_filter["count"] >= 50].rename(columns={"count":"count_target"})

df_filter.sort_values(by="count_target", ascending=False)

,Usage,text,target,count_target
145006,Public,‫کبغػ ہیں۔ کوئی ثھوکب وکیل هؼلوم ہوتب ہے۔‬,tgk,1500
47074,Public,«Ўзбекинвест» экспорт-импорт миллий суғурта ко...,tgk,1500
26719,Public,Янгиликлар » Дунё » “Лавозимини кўтариш орқали...,tgk,1500
26725,Public,وزارت محنت کے مطابق وہ غیر ملکی افراد جنہو ں ن...,tgk,1500
110921,Public,11. Китоб ва Журналнинг муқоваси қаттиқ матери...,tgk,1500
...,...,...,...,...
182602,Public,Maayo nga Pagkari sa GNOME Desktop,hil,58
20764,Public,Ini magatudlo sa iya kon daw ano kadaku sang i...,hil,58
2593,Public,Kay ini amo ang gugma sang Dios nga bantayan t...,hil,58
17062,Public,May tanom man si Uwa nga pinya.,hil,58


In [5]:
# adding new examples from merging multiple texts randomly
df_50_to_500 = df_filter[(50<df_filter["count_target"])&(df_filter["count_target"]<1000)]

df_grouped_by_targets = df_50_to_500.groupby(["target","count_target"])[["text"]].agg(lambda x: " ".join(list(x)).split()).reset_index()

dict_new_texts = dict()

for target, count_target, all_cleaned_text in df_grouped_by_targets.values:
    new_str_list = []
    # we don't want the new texts to be predominant, but also we want to ensure at least 500 total examples
    for _ in range(count_target, max(500, min(1000, round(count_target*1.5))), 1):
        size = random.randint(60, 140)
        new_str = " ".join(random.sample(all_cleaned_text, size))
        new_str_list.append(new_str)

    dict_new_texts[target] = new_str_list

In [6]:
# we now have at least 500 cases examples for each target
df_new_texts = pd.DataFrame(
    [(key, value) for key, values in dict_new_texts.items() for value in values], 
     columns=("target","text")
)

df_new_texts = pd.concat([df_new_texts, df_filter[["target","text"]]])

# count number of targets
count_targets = (df_new_texts.target.value_counts()).reset_index()
df_new_texts = df_new_texts.merge(count_targets, on="target").rename(columns={"count":"count_target"})

df_clean = df_new_texts.copy()
df_clean['cleaned_text'] = df_clean['text'].apply(preprocessor.clean_text)

df_clean.head()

,target,text,count_target,cleaned_text
0,abk,наҟ даҽакы уааԥсан. Ҳахьааиз Уи ҟыт анаҟә Хлан...,750,наҟ даҽакы уааԥсан ҳахьааиз уи ҟыт анаҟә хланҵ...
1,abk,Арҭ Инҵәо Твардовски шәкы ирҭахыз сымала тәаха...,750,арҭ инҵәо твардовски шәкы ирҭахыз сымала тәаха...
2,abk,ихшарагьы банааиуагьы снапсыргәыҵа рхахьы егьы...,750,ихшарагьы банааиуагьы снапсыргәыҵа рхахьы егьы...
3,abk,сыцҳәа. КПК аҭаҳмадцәа Нас иҳәеит иҟаз илеиуаз...,750,сыцҳәа кпк аҭаҳмадцәа нас иҳәеит иҟаз илеиуаз ...
4,abk,Аҳәбақәа алсхьан. Уиадагьы уԥаҵа иҟаз сахьыԥшу...,750,аҳәбақәа алсхьан уиадагьы уԥаҵа иҟаз сахьыԥшуа...


In [14]:
# add new features based on cleaned_texts
class TextAnalyzer:
    def __init__(self, text):
        self.text = text
        self.features = self.extract_features()

    @staticmethod
    def count_vowels_consonants(text):
        vowels = set("aeiouAEIOUáéíóúÁÉÍÓÚàèìòùÀÈÌÒÙäëïöüÄËÏÖÜãõÃÕåÅøØæÆœŒ")
        consonants = set("bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ")

        vowel_count = sum(1 for char in text if char in vowels)
        consonant_count = sum(1 for char in text if char in consonants)

        return vowel_count, consonant_count

    @staticmethod
    def detect_script(text):
        scripts = {}
        for char in text:
            try:
                script = unicodedata.name(char).split()[0]  # Extract script name
                scripts[script] = scripts.get(script, 0) + 1
            except ValueError:
                continue  # Ignore characters without a name
        return max(scripts, key=scripts.get, default="Unknown")  # Handle empty cases

    @staticmethod
    def extract_ngrams(text, n=2):
        """Extract character n-grams"""
        text = re.sub(r"\s+", "", text)  # Remove spaces
        return [text[i:i+n] for i in range(len(text) - n + 1)]

    @staticmethod
    def extract_word_ngrams(text, n=2):
        """Extract word n-grams"""
        words = text.split()
        return [" ".join(words[i:i+n]) for i in range(len(words) - n + 1)]

    def extract_features(self):
        if not isinstance(self.text, str) or len(self.text) == 0:
            return {
                "length": 0,
                "word_count": 0,
                "vowel_count": 0,
                "consonant_count": 0,
                "digit_count": 0,
                "punctuation_count": 0,
                "unique_chars": 0,
                "uppercase_ratio": 0.0,
                "script": "Unknown",
                "most_common_char": None,
                "diacritics_count": 0,
                # "langdetect": "Unknown",
                # "word_bigrams": [],
                "special_chars": []
            }

        vowels, consonants = self.count_vowels_consonants(self.text)
        script = self.detect_script(self.text)
        length = len(self.text)
        word_count = len(self.text.split())
        digit_count = sum(c.isdigit() for c in self.text)
        punctuation_count = sum(1 for c in self.text if c in string.punctuation)
        unique_chars = len(set(self.text))  # Number of unique characters
        uppercase_ratio = sum(1 for c in self.text if c.isupper()) / length if length > 0 else 0

        # Most common character
        char_counts = Counter(self.text)
        most_common_char = char_counts.most_common(1)[0][0] if char_counts else None

        # Diacritics count
        diacritics_count = sum(1 for c in self.text if unicodedata.combining(c) > 0)

        # Language detection (fallback in case of failure)
        # try:
        #     langdetect_lang = detect(self.text)
        # except:
        #     langdetect_lang = "Unknown"

        # # Extract n-grams
        # word_bigrams = self.extract_word_ngrams(self.text, 2)

        # Special characters presence
        special_chars = [c for c in self.text if not c.isalnum() and c not in " "]

        return {
            "length": length,
            "word_count": word_count,
            "vowel_count": vowels,
            "consonant_count": consonants,
            "digit_count": digit_count,
            "punctuation_count": punctuation_count,
            "unique_chars": unique_chars,
            "uppercase_ratio": uppercase_ratio,
            "script": script,
            "most_common_char": most_common_char,
            "diacritics_count": diacritics_count,
            # "langdetect": langdetect_lang,
            # "word_bigrams": word_bigrams,
            "special_chars": special_chars
        }

    def get_features(self):
        return self.features
    
df_new_features = df_clean.copy()

df_new_features["features"] = df_new_features["text"].apply(lambda text: TextAnalyzer(text).get_features())

df_features_expanded = df_new_features["features"].apply(pd.Series)

df_new_features = df_new_features.drop(columns=["features"]).join(df_features_expanded)

df_new_features.head()

df_new_features.to_csv("df_new_features.csv")

### Training

In [15]:
# Define the text column
text_col = "cleaned_text"

numeric_cols = [
    "length", 
    "word_count", 
    "vowel_count", 
    "consonant_count", 
    "digit_count", 
    "punctuation_count", 
    "unique_chars", 
    "uppercase_ratio", 
    "diacritics_count"
]

categorical_cols = [
    "script",          
    "most_common_char",  
    # "langdetect"        
]

X = df_new_features.drop(columns=["target"]) 
y = df_new_features["target"]


# Transforming to TF-IDF
text_transformer = TfidfVectorizer(
    ngram_range=(1,3),  # unigrams, bigrams e trigrams
    max_features=10000, # vocabulary limit
)

# we scale numerical columns
numeric_transformer = StandardScaler()

# one hot encoding for the others
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# combine
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_transformer, text_col),
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop" 
)


pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)

score = pipeline.score(X_test, y_test)
print("Test accuracy:", score)


In [11]:
df_new_features = pd.read_csv("df_new_features.csv")

df = df_new_features.copy()

le = LabelEncoder()

X = df.drop(columns=["target"])
y = df["target"]
y_encoded = le.fit_transform(y)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded 
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [9]:
text_col = "cleaned_text"

numeric_cols = [
    "length",
    "word_count",
    "vowel_count",
    "consonant_count",
    "digit_count",
    "punctuation_count",
    "unique_chars",
    "uppercase_ratio",
    "diacritics_count",
]

# Se você tiver colunas categóricas que façam sentido:
categorical_cols = ["script","most_common_char"] 
# categorical_cols = ["script","most_common_char","special_chars"] 

text_transformer = TfidfVectorizer(
    ngram_range=(1, 2),  # unigrams + bigrams
    max_features=5000,   # limit for vocabulary
)

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_transformer, text_col),
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)


def create_pipeline(estimator):
    return Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", estimator),
    ])


models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    # "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    # "XGB": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42),
}

results = {}

for model_name, model_obj in models.items():
    pipeline = create_pipeline(model_obj)
    pipeline.fit(X_train, y_train)

    # Accuracy
    train_score = pipeline.score(X_train, y_train)
    val_score = pipeline.score(X_val, y_val)

    results[model_name] = (train_score, val_score)

for m, (tr, val) in results.items():
    print(f"{m} -> train: {tr:.4f}, val: {val:.4f}")

# Choosing best model
best_model_name = max(results, key=lambda m: results[m][1])  # maior val_score
best_model_obj = models[best_model_name]

pipeline_best = create_pipeline(best_model_obj)
pipeline_best.fit(X_train, y_train)

print("Best model:", best_model_name)
print("Val Score:", pipeline_best.score(X_val, y_val))
print("Test Score:", pipeline_best.score(X_test, y_test))

NameError: name 'X_train' is not defined

In [ ]:
preproc = pipeline_best["preprocessing"]

tfidf_colnames = preproc.named_transformers_["text"].get_feature_names_out()  # TF-IDF names
num_colnames = numeric_cols
cat_enc = preproc.named_transformers_["cat"]
cat_colnames = cat_enc.get_feature_names_out(categorical_cols)

# Concatenate everything in the same order as it appears in the final pipeline
feature_names = np.concatenate([tfidf_colnames, num_colnames, cat_colnames])

# Access to the classifier's feature importances
classifier = pipeline_best["classifier"]
importances = classifier.feature_importances_  

# Ordenar por importância
indices = np.argsort(importances)[::-1]
sorted_features = [(feature_names[i], importances[i]) for i in indices]

for feat, imp in sorted_features[:30]:
    print(f"{feat}: {imp:.4f}")

r = permutation_importance(
    pipeline_best, X_val, y_val, n_repeats=3, random_state=42
)
perm_importances = r.importances_mean

indices = np.argsort(perm_importances)[::-1]
sorted_features_perm = [(feature_names[i], perm_importances[i]) for i in indices]

print("Permutation Importance (top 30):")
for feat, imp in sorted_features_perm[:30]:
    print(f"{feat}: {imp:.4f}")